In [3]:
# 환경 변수 설정
!export REPO_DIR=/content/drive/MyDrive/ITRC/tab-ddpm  # 레포지토리 경로로 바꿔주세요
!cd $REPO_DIR

# Conda를 설치합니다.
!wget https://repo.anaconda.com/miniconda/Miniconda3-py39_4.10.3-Linux-x86_64.sh
!chmod +x Miniconda3-py39_4.10.3-Linux-x86_64.sh
!./Miniconda3-py39_4.10.3-Linux-x86_64.sh -b -f -p /usr/local

# 새로 설치한 conda를 PATH에 추가
import sys
sys.path.append('/usr/local/lib/python3.9/site-packages')

# 새로 설치한 conda를 활성화
!conda init


# Conda 환경 생성
!conda create -n tddpm python=3.9.7
!conda activate tddpm

# PyTorch 및 필수 패키지 설치
!pip install torch==1.10.1+cu111 -f https://download.pytorch.org/whl/torch_stable.html
!pip install -r /content/drive/MyDrive/ITRC/tab-ddpm/requirements.txt

# 환경 변수 설정
!conda env config vars set PYTHONPATH=${PYTHONPATH}:${REPO_DIR}
!conda env config vars set PROJECT_DIR=${REPO_DIR}


--2025-11-21 02:23:25--  https://repo.anaconda.com/miniconda/Miniconda3-py39_4.10.3-Linux-x86_64.sh
Resolving repo.anaconda.com (repo.anaconda.com)... 104.16.32.241, 104.16.191.158, 2606:4700::6810:20f1, ...
Connecting to repo.anaconda.com (repo.anaconda.com)|104.16.32.241|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 66709754 (64M) [application/x-sh]
Saving to: ‘Miniconda3-py39_4.10.3-Linux-x86_64.sh’

Miniconda3-py39_4.1 100%[===================>]  63.62M  31.3MB/s    in 2.0s    

2025-11-21 02:23:27 (31.3 MB/s) - ‘Miniconda3-py39_4.10.3-Linux-x86_64.sh’ saved [66709754/66709754]

PREFIX=/usr/local
Unpacking payload ...
Solving environment: / - \ | done

## Package Plan ##

  environment location: /usr/local

  added / updated specs:
    - _libgcc_mutex==0.1=main
    - _openmp_mutex==4.5=1_gnu
    - brotlipy==0.7.0=py39h27cfd23_1003
    - ca-certificates==2021.7.5=h06a4308_1
    - certifi==2021.5.30=py39h06a4308_0
    - cffi==1.14.6=py39h400218f_0


In [ ]:
import argparse
import enum
import json
import math
import random
import shutil
import sys
import zipfile
from copy import deepcopy
from pathlib import Path
from typing import Any, Optional, cast
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import pyarrow.csv
import sklearn.datasets
import sklearn.utils
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm


In [ ]:
ArrayDict = dict[str, np.ndarray]
Info = dict[str, Any]

DATA_DIR = Path('/content/drive/MyDrive/ITRC/tab-ddpm/data/adult/adult.csv')
SEED = 0
CAT_MISSING_VALUE = '__nan__'


In [ ]:
class TaskType(enum.Enum):
    REGRESSION = 'regression'
    BINCLASS = 'binclass'
    MULTICLASS = 'multiclass'

def _set_random_seeds():
    random.seed(SEED)
    np.random.seed(SEED)


def _download_file(url: str, path: Path):
    assert not path.exists()
    try:
        print(f'Downloading {url} ...', end='', flush=True)
        urlretrieve(url, path)
    except Exception:
        if path.exists():
            path.unlink()
        raise
    finally:
        print()


def _unzip(path: Path, members: Optional[list[str]] = None) -> None:
    with zipfile.ZipFile(path) as f:
        f.extractall(path.parent, members)


def _start(dirname: str) -> tuple[Path, list[Path]]:
    DATA_DIR = "/content/drive/MyDrive/ITRC/tab-ddpm/data/adult"
    print(f'>>> {dirname}')
    _set_random_seeds()
    dataset_dir = DATA_DIR / dirname
    expected_files = EXPECTED_FILES[dirname]
    if expected_files:
        assert dataset_dir.exists()
        assert set(expected_files) == set(x.name for x in dataset_dir.iterdir())
    else:
        assert not dataset_dir.exists()
        dataset_dir.mkdir()
    return dataset_dir, [dataset_dir / x for x in expected_files]


def _fetch_openml(data_id: int) -> sklearn.utils.Bunch:
    bunch = cast(
        sklearn.utils.Bunch,
        sklearn.datasets.fetch_openml(data_id=data_id, as_frame=True),
    )
    assert not bunch['categories']
    return bunch


def _get_sklearn_dataset(name: str) -> tuple[np.ndarray, np.ndarray]:
    get_data = getattr(sklearn.datasets, f'load_{name}', None)
    if get_data is None:
        get_data = getattr(sklearn.datasets, f'fetch_{name}', None)
    assert get_data is not None, f'No such dataset in scikit-learn: {name}'
    return get_data(return_X_y=True)


def _encode_classification_target(y: np.ndarray) -> np.ndarray:
    assert not str(y.dtype).startswith('float')
    if str(y.dtype) not in ['int32', 'int64', 'uint32', 'uint64']:
        y = LabelEncoder().fit_transform(y)
    else:
        labels = set(map(int, y))
        if sorted(labels) != list(range(len(labels))):
            y = LabelEncoder().fit_transform(y)
    return y.astype(np.int64)


def _make_split(size: int, stratify: Optional[np.ndarray], n_parts: int) -> ArrayDict:
    # n_parts == 3:      all -> train & val & test
    # n_parts == 2: trainval -> train & val
    assert n_parts in (2, 3)
    all_idx = np.arange(size, dtype=np.int64)
    a_idx, b_idx = train_test_split(
        all_idx,
        test_size=0.2,
        stratify=stratify,
        random_state=SEED + (1 if n_parts == 2 else 0),
    )
    if n_parts == 2:
        return cast(ArrayDict, {'train': a_idx, 'val': b_idx})
    a_stratify = None if stratify is None else stratify[a_idx]
    a1_idx, a2_idx = train_test_split(
        a_idx, test_size=0.2, stratify=a_stratify, random_state=SEED + 1
    )
    return cast(ArrayDict, {'train': a1_idx, 'val': a2_idx, 'test': b_idx})


def _apply_split(data: ArrayDict, split: ArrayDict) -> dict[str, ArrayDict]:
    return {k: {part: v[idx] for part, idx in split.items()} for k, v in data.items()}


def _save(
    dataset_dir: Path,
    name: str,
    task_type: TaskType,
    *,
    X_num: Optional[ArrayDict],
    X_cat: Optional[ArrayDict],
    y: ArrayDict,
    idx: Optional[ArrayDict],
    id_: Optional[str] = None,
    id_suffix: str = '--default',
) -> None:
    if id_ is not None:
        assert id_suffix == '--default'
    assert (
        X_num is not None or X_cat is not None
    ), 'At least one type of features must be presented.'
    if X_num is not None:
        X_num = {k: v.astype(np.float32) for k, v in X_num.items()}
    if X_cat is not None:
        X_cat = {k: v.astype(str) for k, v in X_cat.items()}
    if idx is not None:
        idx = {k: v.astype(np.int64) for k, v in idx.items()}
    y = {
        k: v.astype(np.float32 if task_type == TaskType.REGRESSION else np.int64)
        for k, v in y.items()
    }
    if task_type != TaskType.REGRESSION:
        y_unique = {k: set(v.tolist()) for k, v in y.items()}
        assert y_unique['train'] == set(range(max(y_unique['train']) + 1))
        for x in ['val', 'test']:
            assert y_unique[x] <= y_unique['train']
        del x

    info = {
        'name': name,
        'id': (dataset_dir.name + id_suffix) if id_ is None else id_,
        'task_type': task_type.value,
        'n_num_features': (0 if X_num is None else next(iter(X_num.values())).shape[1]),
        'n_cat_features': (0 if X_cat is None else next(iter(X_cat.values())).shape[1]),
    } | {f'{k}_size': len(v) for k, v in y.items()}
    if task_type == TaskType.MULTICLASS:
        info['n_classes'] = len(set(y['train']))
    (dataset_dir / 'info.json').write_text(json.dumps(info, indent=4))

    for data_name in ['X_num', 'X_cat', 'y', 'idx']:
        data = locals()[data_name]
        if data is not None:
            for k, v in data.items():
                np.save(dataset_dir / f'{data_name}_{k}.npy', v)
    (dataset_dir / 'READY').touch()
    print('Done\n')


In [ ]:
# CSV 파일을 읽어 데이터프레임으로 로드
data = pd.read_csv('/content/drive/MyDrive/ITRC/tab-ddpm/data/adult/adult.csv')

# 특정 열 선택
column_name = 'label'  # 변환하려는 열의 이름으로 수정 필요

data = pd.DataFrame(data)

# 열의 값이 'value1'과 'value2'인 경우를 0과 1로 매핑
data[column_name] = data[column_name].map({'<=50K': 0, '>50K': 1})

print(data)

       age  workclass  fnlwgt     education  education-num  \
0       27    Private  177119  Some-college             10   
1       27    Private  216481     Bachelors             13   
2       25    Private  256263    Assoc-acdm             12   
3       46    Private  147640       5th-6th              3   
4       45    Private  172822          11th              7   
...    ...        ...     ...           ...            ...   
32556   43  Local-gov   33331       Masters             14   
32557   44    Private   98466          10th              6   
32558   23    Private   45317  Some-college             10   
32559   45  Local-gov  215862     Doctorate             16   
32560   25    Private  186925  Some-college             10   

           marital-status        occupation   relationship  \
0                Divorced      Adm-clerical      Unmarried   
1           Never-married    Prof-specialty  Not-in-family   
2      Married-civ-spouse             Sales        Husband   
3      

In [ ]:
def adult():
    # Get the file here: https://www.kaggle.com/shrutimechlearn/churn-modelling
    dataset_dir = Path('/content/drive/MyDrive/ITRC/tab-ddpm/data/adult')
    df = pd.read_csv('/content/drive/MyDrive/ITRC/tab-ddpm/data/adult/adult.csv')

    #df = df.drop(columns=['RowNumber', 'CustomerId', 'Surname'])
    #df['Gender'] = df['Gender'].astype('category').cat.codes.values.astype(np.int64) category를 숫자로 표현
    y_all = (df.pop('income') == '>50K').values.astype('int64')
    num_columns = ['fnlwgt','education-num','capital-gain','capital-loss','hours-per-week']
    cat_columns = ['age','workclass','education','marital-status','occupation','relationship',
                        'race','sex','native-country']
    assert set(num_columns) | set(cat_columns) == set(df.columns.tolist())
    X_num_all = df[num_columns].values
    X_cat_all = df[cat_columns].values
    idx = _make_split(len(df), y_all, 3)

    _save(
        dataset_dir,
        'adult',
        TaskType.BINCLASS,
        **_apply_split(
            {'X_num': X_num_all, 'X_cat': X_cat_all, 'y': y_all},
            idx,
        ),
        idx=idx,
    )

In [ ]:
adult()

Done



In [ ]:
!python '/content/drive/MyDrive/ITRC/tab-ddpm/scripts/pipeline.py' --config '/content/drive/MyDrive/ITRC/tab-ddpm/exp/adult/config.toml' --train --sample


[73  9 16  7 15  6  5  2 41]
179
{'num_classes': 2, 'is_y_cond': True, 'rtdl_params': {'d_layers': [256, 256], 'dropout': 0.0}, 'd_in': 179}
mlp
Step 500/1000 MLoss: 1.4091 GLoss: 0.9042 Sum: 2.3133
Step 1000/1000 MLoss: 1.3666 GLoss: 0.7636 Sum: 2.1302
<generator object Module.parameters at 0x783fe98dd6d0>
mlp
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0

Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0

Sample timestep    0
Sample timestep    0
Discrete cols: [0, 1, 2, 3, 4]
Num

In [ ]:
cat_train = np.load('/content/drive/MyDrive/ITRC/tab-ddpm/data/adult/y_train.npy', allow_pickle=True)
cat = pd.DataFrame(cat_train)

In [ ]:

cat_train = np.load('/content/drive/MyDrive/ITRC/tab-ddpm/exp/adult/check/X_cat_train.npy', allow_pickle=True)
cat = pd.DataFrame(cat_train)

# CSV 파일로 저장
cat_file = f'/content/drive/MyDrive/ITRC/tab-ddpm/exp/adult/check/adult1_tabddpm_X_cat_train.csv'
cat.to_csv(cat_file, index=False, header=False)

num_train = np.load('/content/drive/MyDrive/ITRC/tab-ddpm/exp/adult/check/X_num_train.npy', allow_pickle=True)
num = pd.DataFrame(num_train)

# CSV 파일로 저장
num_file = f'/content/drive/MyDrive/ITRC/tab-ddpm/exp/adult/check/adult1_tabddpm_X_num_train.csv'
num.to_csv(num_file, index=False, header=False)



y_train = np.load('/content/drive/MyDrive/ITRC/tab-ddpm/exp/adult/check/y_train.npy', allow_pickle=True)
y = pd.DataFrame(y_train)

# CSV 파일로 저장
y_file = f'/content/drive/MyDrive/ITRC/tab-ddpm/exp/adult/check/adult1_tabddpm_y_train.csv'
y.to_csv(y_file, index=False, header=False)